## Příprava prostředí

Před spuštěním notebooku nainstalujte závislosti z kořenového adresáře projektu:

```bash
pip install -r requirements.txt
```

# Analýza detekce mytí rukou - Výsledky Baseline modelu

Tento notebook dokumentuje proces vývoje a vyhodnocení základního algoritmu pro detekci mytí rukou v rámci bakalářské práce.

## 1. Cíl projektu
Automatická detekce hygienických návyků (mytí rukou) ve výrobním prostředí. Postupujeme od jednoduchého pohybu k pokročilé AI detekci.

## 2. Příprava dat (Ground Truth)
Bylo vytvořeno a ručně anotováno celkem **162 video klipů**.
- Zdroj: Průmyslová kamera nad umyvadlem.
- Anotace: Vlastní nástroj `src/annotate.py`.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Přidání kořene projektu do PYTHONPATH pro import config
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import OUTPUTS_DIR

# Načtení anotací
annotations_path = OUTPUTS_DIR / "annotations.json"
with open(annotations_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

total_clips = len(data)
clips_with_wash = sum(1 for v in data.values() if len(v['events']) > 0)

print(f"Celkový počet klipů: {total_clips}")
print(f"Klipy s detekovaným mytím: {clips_with_wash}")
print(f"Klipy bez aktivity: {total_clips - clips_with_wash}")

## 3. Výsledky Baseline modelu (Detekce na základě pohybu)

Algoritmus používá odečítání pozadí (MOG2) v definované oblasti zájmu (ROI).

In [ ]:
results = {
    "Metrika": ["Precision", "Recall", "F1-Score", "Mean IoU"],
    "Hodnota": [0.4706, 0.7442, 0.5766, 0.3940]
}
df_metrics = pd.DataFrame(results)

plt.figure(figsize=(10, 6))
plt.bar(df_metrics["Metrika"], df_metrics["Hodnota"], color=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
plt.ylim(0, 1)
plt.title("Výkonnost Baseline modelu (Detekce pohybu)")
plt.ylabel("Skóre (0-1)")
for i, v in enumerate(df_metrics["Hodnota"]):
    plt.text(i, v + 0.02, str(v), color='black', fontweight='bold', ha='center')
plt.show()

## 4. Portování na AI modely a finální výsledky

V této části porovnáváme vývoj od jednoduché detekce pohybu k pokročilému AI modelu. Soap Trigger detektor dosáhl nejlepších výsledků díky fine-tuningu parametrů (min-duration 3s) a čištění datasetu.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison_results = {
    "Detector": ["1. Baseline (MOG2)", "2. MediaPipe Hands", "3. Soap Trigger (Base)", "4. Soap Trigger (Final)"],
    "Precision": [0.5333, 0.5091, 0.6250, 0.8710],
    "Recall": [0.7442, 0.6512, 0.5814, 0.6279],
    "F1-Score": [0.6214, 0.5714, 0.6024, 0.7297],
    "Mean IoU": [0.3940, 0.4009, 0.6645, 0.6610]
}

df_comp = pd.DataFrame(comparison_results)
df_comp.set_index("Detector", inplace=True)

print("Evoluce přesnosti detekce mytí rukou (Fine-tuningem zvýšena Precision o 25 %):")
display(df_comp)

ax = df_comp.plot(kind="bar", figsize=(12, 7), color=['#66b3ff', '#ff9999', '#99ff99', '#ffcc99'])
plt.title("Postupný posun výkonnosti detektorů")
plt.ylabel("Skóre")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc='lower right')

# Add values on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.tight_layout()
plt.show()

## 4. Analýza chyb a další kroky

**Zjištění:**
- Model má vysoký **Recall (74%)** - většinu mytí zachytí.
- Model má nízkou **Precision (47%)** - polovina hlášení jsou falešné poplachy (stíny, odlesky vody).

**Realizované vylepšení — MediaPipe Hands detektor:**
Implementován AI-enhanced detektor (`mediapipe_detector.py`), který kombinuje MOG2 detekci pohybu s detekcí rukou pomocí Google MediaPipe Hands. Událost je zaznamenána pouze pokud je v ROI detekován pohyb **A ZÁROVEŇ** jsou přítomny lidské ruce. Tím se eliminují falešné poplachy způsobené neživými objekty.

## 5. Plán: Compliance statistiky pro výrobní provoz

### Varianta A — Anonymní compliance monitoring (v realizaci)
Propojení wash událostí s průchody osob přes exit zónu (YOLO person detector):
- **Compliance rate** — % průchodů s předchozím mytím rukou
- **Compliance per směna** — ranní / odpolední / noční
- **Průměrná délka mytí** — WHO doporučuje ≥20s
- **Trend v čase** — např. měření efektu školení

### Varianta B — Pseudonymní tracking (připraveno k rozšíření)
Pro regulované provozy (farmacie, potravinářství) je možnost rozšíření o multi-object tracking (ByteTrack/DeepSORT + YOLO), který umožní sledování na úrovni pseudonymních ID bez rozpoznávání obličejů. Viz `README.md` sekce „Compliance statistiky".

## 6. Hloubková analýza chyb (Error Analysis)

Soap Trigger detektor má vysokou Precision (85 %), ale propouští **21 z 43** skutečných událostí.
Tato sekce identifikuje a kategorizuje všechny False Negatives (FN) a False Positives (FP),
abychom pochopili *proč* model selhává a kde je prostor pro zlepšení.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from config import OUTPUTS_DIR, LABELED_DIR

# Load GT
with open(OUTPUTS_DIR / "annotations.json", "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# Load per-clip eval results
eval_df = pd.read_csv(OUTPUTS_DIR / "eval_soap_trigger.csv")

# Classify clips
fn_clips = []  # GT exists but not detected (or partially)
fp_clips = []  # Detection exists but no GT
tp_clips = []  # Correct matches
tn_clips = []  # Correctly empty

for _, row in eval_df.iterrows():
    name = row['video']
    gt_n = row['gt_count']
    pred_n = row['pred_count']
    match_n = row['match']
    
    has_fn = gt_n > match_n
    has_fp = pred_n > match_n
    
    if has_fn:
        fn_clips.append({
            'clip': name,
            'gt_events': gt_n,
            'detected': match_n,
            'missed': gt_n - match_n,
            'false_alarms': pred_n - match_n
        })
    if has_fp:
        fp_clips.append({
            'clip': name,
            'gt_events': gt_n,
            'false_alarms': pred_n - match_n
        })
    if match_n > 0 and not has_fn and not has_fp:
        tp_clips.append(name)
    if gt_n == 0 and pred_n == 0:
        tn_clips.append(name)

print(f"Celkový přehled ({len(eval_df)} klipů):")
print(f"  ✓ True Positive (správně detekováno):  {len(tp_clips)} klipů")
print(f"  ✓ True Negative (správně prázdné):     {len(tn_clips)} klipů")
print(f"  ✗ False Negative (propuštěné události): {len(fn_clips)} klipů")
print(f"  ✗ False Positive (falešné poplachy):    {len(fp_clips)} klipů")

### 6.1 False Negatives — propuštěné události

Následující klipy obsahují GT události, které detektor **nezachytil**:

In [ ]:
if fn_clips:
    df_fn = pd.DataFrame(fn_clips)
    df_fn = df_fn.sort_values('missed', ascending=False)
    print("Klipy s propuštěnými událostmi (FN):")
    display(df_fn)
    
    print(f"\nCelkem propuštěno {df_fn['missed'].sum()} událostí v {len(df_fn)} klipech.")
    print(f"Průměrně {df_fn['missed'].mean():.1f} propuštěných událostí na problémový klip.")
else:
    print("Žádné False Negatives! 🎉")

### 6.2 False Positives — falešné poplachy

Následující klipy obsahují detekce, které **neodpovídají žádné GT události**:

In [ ]:
if fp_clips:
    df_fp = pd.DataFrame(fp_clips)
    df_fp = df_fp.sort_values('false_alarms', ascending=False)
    print("Klipy s falešnými poplachy (FP):")
    display(df_fp)
else:
    print("Žádné False Positives! 🎉")

### 6.3 Timeline vizualizace: GT vs. Predikce

Pro každý problémový klip zobrazujeme časovou osu s Ground Truth (zelená) a predikcí detektoru (červená).
Kde se pruhy překrývají = správná detekce (TP). Zelený pruh bez červeného = FN. Červený bez zeleného = FP.

In [ ]:
from evaluate import evaluate_performance, calculate_iou
from soap_trigger_detector import detect_wash_events as soap_detect
from roi_select import load_roi
from config import DEFAULT_ROI_PATH, DetectionParams

# Load ROI and soap zones
roi_data = load_roi(DEFAULT_ROI_PATH)
soap_zones = roi_data.pop('soap_zones', None)
if soap_zones is None:
    single = roi_data.pop('soap_zone', None)
    if single is not None:
        soap_zones = [single]
else:
    roi_data.pop('soap_zone', None)
roi = roi_data

params = DetectionParams()

def plot_timeline(clip_name, gt_events, pred_events, clip_duration=20.0):
    """Plot GT vs prediction timeline for a single clip."""
    fig, ax = plt.subplots(figsize=(14, 1.8))
    
    # GT events (green)
    for ev in gt_events:
        ax.barh(1, ev['end_sec'] - ev['start_sec'], left=ev['start_sec'],
                height=0.4, color='#2ecc71', alpha=0.85, edgecolor='#27ae60', linewidth=1.5)
    
    # Predicted events (red)
    for ev in pred_events:
        ax.barh(0, ev['end_sec'] - ev['start_sec'], left=ev['start_sec'],
                height=0.4, color='#e74c3c', alpha=0.85, edgecolor='#c0392b', linewidth=1.5)
    
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Predikce', 'Ground Truth'])
    ax.set_xlim(0, clip_duration)
    ax.set_xlabel('Čas (s)')
    ax.set_title(clip_name, fontweight='bold', fontsize=11)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Select problem clips (all FN + all FP clips)
problem_clips = set()
for item in fn_clips:
    problem_clips.add(item['clip'])
for item in fp_clips:
    problem_clips.add(item['clip'])

print(f"Zobrazuji timeline pro prvních {min(MAX_PREVIEW, len(problem_clips))} z {len(problem_clips)} problémových klipů:")

# Show max 5 clips as preview (change MAX_PREVIEW to see more)
MAX_PREVIEW = 5
for clip_name in sorted(problem_clips)[:MAX_PREVIEW]:
    clip_path = LABELED_DIR / clip_name
    if not clip_path.exists():
        print(f"  ⚠ {clip_name} nenalezen, přeskakuji.")
        continue
    
    gt_events = gt_data.get(clip_name, {}).get('events', [])
    
    # Run detector
    try:
        pred_df = soap_detect(
            video_path=str(clip_path),
            roi=roi,
            soap_zones=soap_zones,
            params=params,
            show_preview=False,
        )
        pred_events = pred_df.to_dict('records') if not pred_df.empty else []
    except Exception as e:
        print(f"  ✗ Chyba u {clip_name}: {e}")
        continue
    
    plot_timeline(clip_name, gt_events, pred_events)

### 6.4 Distribuce chyb napříč detektory

Srovnání počtu FP a FN pro každý detektor — ukazuje, jak se charakter chyb mění s evolucí modelu.

In [ ]:
# Load all summaries
summaries = {}
for det in ['baseline', 'mediapipe', 'soap_trigger']:
    path = OUTPUTS_DIR / f'eval_{det}_summary.json'
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            summaries[det] = json.load(f)

if summaries:
    labels = []
    fps = []
    fns = []
    tps = []
    
    name_map = {
        'baseline': '1. Baseline (MOG2)',
        'mediapipe': '2. MediaPipe Hands',
        'soap_trigger': '3. Soap Trigger',
    }
    
    for det in ['baseline', 'mediapipe', 'soap_trigger']:
        if det in summaries:
            s = summaries[det]
            labels.append(name_map.get(det, det))
            fps.append(s['fp'])
            fns.append(s['fn'])
            tps.append(s['tp'])
    
    x = np.arange(len(labels))
    width = 0.55
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bars_tp = ax.bar(x, tps, width, label='True Positives (TP)', color='#2ecc71', edgecolor='white')
    bars_fp = ax.bar(x, fps, width, bottom=tps, label='False Positives (FP)', color='#e74c3c', edgecolor='white')
    bars_fn = ax.bar(x, fns, width, bottom=[t+f for t,f in zip(tps, fps)],
                    label='False Negatives (FN)', color='#f39c12', edgecolor='white')
    
    ax.set_ylabel('Počet událostí')
    ax.set_title('Distribuce TP / FP / FN napříč detektory', fontweight='bold', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bars in [bars_tp, bars_fp, bars_fn]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2., bar.get_y() + h/2.,
                        f'{int(h)}', ha='center', va='center', fontweight='bold', color='white', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print("\nKlíčové poznatky:")
    print("  • Baseline: hodně TP (32), ale i hodně FP (28) — detekuje skoro vše, ale i falešně.")
    print("  • MediaPipe: AI vrstva nepomohla eliminovat FP (27) a ještě zvýšila FN (15).")
    print("  • Soap Trigger: dramatický pokles FP (jen 4!), ale nárůst FN na 21.")
    print("  → Hlavní optimalizační cíl: SNÍŽIT FN při zachování nízkého FP.")

### 6.5 Analýza propuštěných událostí — jsou krátké nebo dlouhé?

Zkoumáme, zda jsou propuštěné události (FN) systematicky kratší/delší než správně zachycené (TP).

In [ ]:
# Categorize all GT events as TP or FN based on eval results
tp_durations = []
fn_durations = []

for _, row in eval_df.iterrows():
    name = row['video']
    gt_evs = gt_data.get(name, {}).get('events', [])
    
    if gt_data.get(name, {}).get('exclude', False):
        continue
    
    match_n = row['match']
    gt_n = row['gt_count']
    
    # Sort GT events by duration for analysis
    for ev in gt_evs:
        dur = ev['end_sec'] - ev['start_sec']
        if match_n > 0:
            tp_durations.append(dur)
            match_n -= 1
        else:
            fn_durations.append(dur)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
all_durs = tp_durations + fn_durations
bins = np.linspace(0, max(all_durs) if all_durs else 20, 15)

axes[0].hist(tp_durations, bins=bins, alpha=0.7, color='#2ecc71', label=f'TP ({len(tp_durations)})', edgecolor='white')
axes[0].hist(fn_durations, bins=bins, alpha=0.7, color='#e74c3c', label=f'FN ({len(fn_durations)})', edgecolor='white')
axes[0].set_xlabel('Délka GT události (s)')
axes[0].set_ylabel('Počet')
axes[0].set_title('Distribuce délek: zachycené (TP) vs. propuštěné (FN)', fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot
bp_data = []
bp_labels = []
if tp_durations:
    bp_data.append(tp_durations)
    bp_labels.append(f'TP (n={len(tp_durations)})')
if fn_durations:
    bp_data.append(fn_durations)
    bp_labels.append(f'FN (n={len(fn_durations)})')

bp = axes[1].boxplot(bp_data, labels=bp_labels, patch_artist=True,
                     boxprops=dict(facecolor='#ecf0f1'),
                     medianprops=dict(color='#e74c3c', linewidth=2))
if len(bp['boxes']) >= 1:
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][0].set_alpha(0.5)
if len(bp['boxes']) >= 2:
    bp['boxes'][1].set_facecolor('#e74c3c')
    bp['boxes'][1].set_alpha(0.5)

axes[1].set_ylabel('Délka události (s)')
axes[1].set_title('Box plot: TP vs. FN délky', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistiky délek GT událostí:")
if tp_durations:
    print(f"  TP: průměr={np.mean(tp_durations):.1f}s, medián={np.median(tp_durations):.1f}s, min={min(tp_durations):.1f}s, max={max(tp_durations):.1f}s")
if fn_durations:
    print(f"  FN: průměr={np.mean(fn_durations):.1f}s, medián={np.median(fn_durations):.1f}s, min={min(fn_durations):.1f}s, max={max(fn_durations):.1f}s")

### 6.6 Shrnutí Error Analysis

**Klíčová zjištění:**

1. **FP problém je vyřešen** — Soap Trigger má jen 4 FP (oproti 28 u Baseline). Tento přístup funguje.
2. **FN je hlavní slabina** — 21 z 43 událostí není detekováno. Tři hlavní příčiny:
   - Ruka se dotýká mýdla mimo definovanou soap zónu (nebo mýdlo nepoužije)
   - MediaPipe nedetekuje ruku (mokré/lesklé ruce, nízký kontrast)
   - Událost se rozpadne na dvě krátké části, obě odfiltrovány min-duration filtrem
3. **Další kroky** → Temporal smoothing (§5), Event merging (§6), Grid Search (§4)

## 7. Ablační studie (Ablation Study)

Systematické vyhodnocení přínosu každé komponenty detekčního pipeline.
Postupně přidáváme složky a měříme, jak každá ovlivňuje metriky.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import OUTPUTS_DIR, LABELED_DIR, DEFAULT_ROI_PATH, DetectionParams
from evaluate import evaluate_performance

# 1. Baseline (MOG2 only) — already evaluated
# 2. MediaPipe Hands (MOG2 + hand detection) — already evaluated
# 3. Soap Trigger WITHOUT post-filters (min-duration=0, min-sink=0)
# 4. Soap Trigger WITH post-filters (current best)

print('=== Ablační studie: spouštím Soap Trigger BEZ post-filtrů... ===')
print('(Toto může trvat několik minut.)\n')

params_no_filter = DetectionParams()
params_no_filter.soap_min_event_duration_sec = 0.0  # disable min-duration filter
params_no_filter.soap_min_sink_time_sec = 0.0       # disable min-sink filter

summary_no_filter = evaluate_performance(
    iou_threshold=0.1,
    detector_name='soap_trigger',
    params=params_no_filter
)

In [ ]:
# Load pre-computed summaries for baseline and mediapipe
summaries = {}
for det in ['baseline', 'mediapipe']:
    p = OUTPUTS_DIR / f'eval_{det}_summary.json'
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            summaries[det] = json.load(f)

# Load current soap_trigger summary (with filters)
p = OUTPUTS_DIR / 'eval_soap_trigger_summary.json'
if p.exists():
    with open(p, 'r', encoding='utf-8') as f:
        summaries['soap_trigger'] = json.load(f)

# Add no-filter result
summaries['soap_no_filter'] = summary_no_filter

# Build ablation table
rows = []
configs = [
    ('baseline', 'A) MOG2 Baseline', 'Odečítání pozadí + hystereze'),
    ('mediapipe', 'B) + MediaPipe Hands', 'A + detekce rukou (filtr FP)'),
    ('soap_no_filter', 'C) + Soap Trigger', 'B + trigger zóna u mýdla (bez post-filtrů)'),
    ('soap_trigger', 'D) + Post-filtry', 'C + min-duration + min-sink filtr'),
]

for key, name, desc in configs:
    s = summaries.get(key, {})
    rows.append({
        'Varianta': name,
        'Popis': desc,
        'Precision': s.get('precision', 0),
        'Recall': s.get('recall', 0),
        'F1-Score': s.get('f1_score', 0),
        'Mean IoU': s.get('mean_iou', 0),
        'TP': s.get('tp', 0),
        'FP': s.get('fp', 0),
        'FN': s.get('fn', 0),
    })

df_ablation = pd.DataFrame(rows)
print('Ablační studie — přínos jednotlivých komponent:\n')
display(df_ablation)

In [ ]:
# Ablation bar chart
metrics = ['Precision', 'Recall', 'F1-Score', 'Mean IoU']
variants = df_ablation['Varianta'].tolist()
x = np.arange(len(variants))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, metric in enumerate(metrics):
    vals = df_ablation[metric].tolist()
    bars = ax.bar(x + i * width, vals, width, label=metric, color=colors[i], edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylabel('Skóre')
ax.set_title('Ablační studie: postupný přínos komponent', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(variants, fontsize=9)
ax.legend(loc='upper left')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nInterpretace:')
print('  A→B: MediaPipe přidala detekci rukou, ale FP zůstaly vysoké (ruce přítomny i mimo mytí).')
print('  B→C: Soap trigger dramaticky snížil FP díky domain-specific trigger zóně.')
print('  C→D: Post-filtry (min-duration) dále snížily FP za cenu mírného poklesu Recall.')

## 8. IoU Threshold Sensitivity Analysis

Jak se metriky mění při různých IoU thresholdech?
Nízký threshold (0.05) = stačí jakýkoliv překryv. Vysoký (0.5) = predikce musí přesně sedět.

In [ ]:
from evaluate import evaluate_performance

iou_thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]
detectors = ['baseline', 'mediapipe', 'soap_trigger']
det_labels = {
    'baseline': 'Baseline (MOG2)',
    'mediapipe': 'MediaPipe Hands',
    'soap_trigger': 'Soap Trigger',
}

# Collect results for each detector × threshold
iou_results = {det: {'precision': [], 'recall': [], 'f1': [], 'iou': []} for det in detectors}

print('Spouštím evaluaci pro různé IoU thresholds...')
print('(Soap Trigger běží pro každý threshold → trvá déle.)\n')

for det in detectors:
    for iou_t in iou_thresholds:
        print(f'  {det_labels[det]}, IoU ≥ {iou_t}...', end=' ')
        s = evaluate_performance(iou_threshold=iou_t, detector_name=det)
        iou_results[det]['precision'].append(s['precision'])
        iou_results[det]['recall'].append(s['recall'])
        iou_results[det]['f1'].append(s['f1_score'])
        iou_results[det]['iou'].append(s['mean_iou'])
        print(f'F1={s["f1_score"]:.3f}')

print('\nHotovo!')

In [ ]:
# IoU sensitivity plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'baseline': '#3498db', 'mediapipe': '#f39c12', 'soap_trigger': '#2ecc71'}
markers = {'baseline': 'o', 'mediapipe': 's', 'soap_trigger': 'D'}

metric_names = [('precision', 'Precision'), ('recall', 'Recall'), ('f1', 'F1-Score')]

for ax, (metric_key, metric_label) in zip(axes, metric_names):
    for det in detectors:
        ax.plot(iou_thresholds, iou_results[det][metric_key],
                marker=markers[det], label=det_labels[det],
                color=colors[det], linewidth=2, markersize=8)
    ax.set_xlabel('IoU Threshold')
    ax.set_ylabel(metric_label)
    ax.set_title(f'{metric_label} vs. IoU Threshold', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(iou_thresholds)

plt.tight_layout()
plt.show()

print('Interpretace:')
print('  • Pokud křivka F1 klesá prudce s rostoucím IoU → detekce jsou sice správné,')
print('    ale časově nepřesné (špatný začátek/konec události).')
print('  • Stabilní křivka = model nejen detekuje, ale i přesně lokalizuje události.')